# Гауссовские процессы: регрессия с честной оценкой неопределённости

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Гауссовские процессы».

## Что делает метод

Гауссовский процесс — это гибкая байесовская модель регрессии. Он задаёт распределение над функциями: любая конечная выборка значений функции имеет совместное нормальное распределение. После наблюдения данных модель даёт для каждой точки не только прогноз, но и обоснованную меру неопределённости.

Поведение модели — гладкость, периодичность, масштаб изменчивости — определяется ядерной функцией. Метод применим при малых и среднеразмерных выборках, когда важна оценка неопределённости. В этом блокноте мы построим регрессию гауссовским процессом и сравним ядра. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы измеряете концентрацию загрязнителя в нескольких точках поля. В точках, где измерения есть, вы знаете значение точно. Но что происходит между ними? Очевидно: если две точки близко — вероятно, значения тоже близкие; если далеко — связь слабее.

Гауссовский процесс формализует эту идею:

1. **Вместо одной кривой** — распределение над всеми правдоподобными кривыми. Каждый «образец» из этого распределения — это одна возможная реализация функции, согласующаяся с данными.

2. **Ядро** (kernel, ковариационная функция) — математическое описание того, насколько похожи значения в двух близких точках. Выбрать ядро — значит сказать: «я предполагаю, что моя функция гладкая» или «я допускаю резкие изменения».

3. **Прогноз в новой точке** — это среднее по всем правдоподобным кривым. **Неопределённость** — разброс этих кривых: вблизи данных они сходятся (мы уверены), вдали от данных расходятся (мы не уверены).

Главное достоинство: в отличие от нейросетей или обычной регрессии, гауссовский процесс не просто выдаёт прогноз, но и **честно сообщает, насколько он в нём уверен**. Это критически важно, когда измерения дорогие и нельзя позволить себе множество проверочных точек.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем зашумлённые наблюдения гладкой функции, оставив пробел в середине диапазона. Пробел нужен, чтобы увидеть, как метод честно показывает рост неопределённости там, где данных нет.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

def true_function(x):
    """Гладкая функция, которую модель должна восстановить."""
    return np.sin(1.5 * x) + 0.3 * x

# наблюдения с пробелом в диапазоне [1, 3]
x_obs = np.concatenate([rng.uniform(-5, 1, 12),
                        rng.uniform(3, 5, 10)])
x_obs = np.sort(x_obs)
y_obs = true_function(x_obs) + rng.normal(scale=0.2, size=x_obs.size)

print(f'Наблюдений: {x_obs.size}; пробел в данных на отрезке [1, 3].')
pd.DataFrame({'x': x_obs.round(2), 'y': y_obs.round(2)}).head()

## 4. Применение метода

Обучим гауссовский процесс с помощью `GaussianProcessRegressor` из `scikit-learn`.

**Ядро RBF** (Radial Basis Function, или «гауссово ядро») — задаёт предположение о гладкости: близкие точки дают близкие значения, и связь убывает экспоненциально с расстоянием. Параметр `length_scale` — «длина корреляции»: на каком расстоянии точки ещё считаются «близкими». При малом `length_scale` функция может меняться быстро; при большом — она очень гладкая и медленно меняющаяся.

**`WhiteKernel`** — добавляет компонент независимого шума измерений. Его параметр `noise_level` оценивается из данных и соответствует дисперсии шума.

**`ConstantKernel`** — масштабирующий множитель, позволяющий модели подобрать амплитуду изменений.

Все параметры ядра подбираются автоматически **максимизацией маргинального правдоподобия** — критерия, отвечающего на вопрос: «насколько хорошо эта модель в целом объясняет наблюдения?». Параметр `n_restarts_optimizer=5` означает, что подбор запускается из 5 разных стартовых точек, чтобы избежать локальных оптимумов.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (RBF, WhiteKernel,
                                              ConstantKernel)

X = x_obs.reshape(-1, 1)
kernel = (ConstantKernel(1.0) * RBF(length_scale=1.0)
          + WhiteKernel(noise_level=0.1))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                              n_restarts_optimizer=5,
                              random_state=42)
gp.fit(X, y_obs)
print('Гауссовский процесс обучен.')
print('Подобранное ядро:', gp.kernel_)

### Прогноз с полосой неопределённости

Построим прогноз на плотной сетке. Метод `predict(return_std=True)` возвращает два массива:
- `y_mean` — **апостериорное среднее**: наилучшая оценка функции в каждой точке.
- `y_std` — **стандартное отклонение прогноза**: мера неопределённости. Полоса ±2 стандартных отклонения охватывает истинное значение примерно в 95 % случаев.

Обратите внимание на пробел в данных на отрезке [1, 3]: именно там полоса должна расшириться — это честная реакция модели на отсутствие информации.

In [ ]:
import matplotlib.pyplot as plt

x_grid = np.linspace(-6, 6, 300).reshape(-1, 1)
y_mean, y_std = gp.predict(x_grid, return_std=True)

fig, ax = plt.subplots(figsize=(10.0, 5.6))
ax.plot(x_grid, true_function(x_grid.ravel()),
        color=VIZ['series'][2], linestyle='--',
        label='истинная функция')
ax.plot(x_grid, y_mean, color=VIZ['series'][0],
        label='прогноз (апостериорное среднее)')
ax.fill_between(x_grid.ravel(), y_mean - 2 * y_std,
                y_mean + 2 * y_std, color=VIZ['series'][0],
                alpha=0.2, label='интервал +/- 2 ст. отклонения')
ax.scatter(x_obs, y_obs, color=VIZ['series'][3], zorder=5,
           label='наблюдения')
ax.set_xlabel('Аргумент x')
ax.set_ylabel('Значение функции')
ax.set_title('Регрессия гауссовским процессом')
ax.legend()
fig.tight_layout()
plt.show()

**Как читать этот график.**

Точки — наблюдения (с шумом). Синяя линия — апостериорное среднее (лучший прогноз). Голубая полоса — неопределённость: ±2 стандартных отклонения. Пунктирная линия — истинная функция, которую модель не видела.

Что важно проверить:
1. В области [-5, 1] и [3, 5], где есть данные, полоса должна быть узкой.
2. В пробеле [1, 3] полоса должна заметно расшириться — модель честно говорит «не знаю».
3. За пределами [-5, 5] полоса ещё шире — это экстраполяция, которой не стоит доверять.
4. Истинная кривая должна почти всегда лежать внутри полосы — это признак хорошо откалиброванной модели.

### Образцы функций из апостериорного распределения

Гауссовский процесс задаёт **распределение над функциями** — не одну кривую, а целое семейство правдоподобных зависимостей. Извлечём несколько образцов: каждый — одна «гипотеза» о форме функции, согласующаяся со всеми наблюдениями.

Метод `sample_y` рисует случайные реализации функции из апостериорного распределения. Все они проходят близко к наблюдениям и расходятся в области пробела — это наглядно показывает структуру неопределённости.

In [ ]:
samples = gp.sample_y(x_grid, n_samples=6, random_state=1)

fig, ax = plt.subplots(figsize=(10.0, 5.6))
for i in range(samples.shape[1]):
    ax.plot(x_grid, samples[:, i], linewidth=1.3, alpha=0.9)
ax.scatter(x_obs, y_obs, color=VIZ['series'][3], zorder=5,
           label='наблюдения')
ax.set_xlabel('Аргумент x')
ax.set_ylabel('Значение функции')
ax.set_title('Образцы функций из апостериорного распределения')
ax.legend()
fig.tight_layout()
plt.show()

**Как читать этот график.**

Каждая цветная линия — одна возможная функция, согласующаяся с данными. Там, где данные есть (левая и правая части), все линии почти совпадают — модель уверена. В пробеле [1, 3] линии расходятся — это и есть неопределённость, выраженная наглядно. Количество «разумных» гипотез о форме функции в пробеле — это мера нашего незнания.

### Сравнение ядер

**Ядро** — это сердце гауссовского процесса. Выбрать ядро — значит сформулировать предположение о характере изменения функции. Разные ядра дают разные формы неопределённости и разные стили прогнозов.

- **RBF** (Radial Basis Function): предполагает бесконечно гладкую функцию. Если функция физически должна быть гладкой — хороший выбор. Недостаток: может быть чрезмерно гладким для реальных данных.
- **Matern с nu=1.5**: предполагает один раз дифференцируемую функцию. Более реалистичен для большинства физических задач, где функции не обязаны быть «идеально гладкими».

Ниже оба ядра применяются к одним данным — смотрите на разницу в прогнозах, особенно в пробеле.

In [ ]:
from sklearn.gaussian_process.kernels import Matern

kernels = {
    'RBF (гладкое)': ConstantKernel(1.0) * RBF(1.0)
                     + WhiteKernel(0.1),
    'Matern (nu=1.5)': ConstantKernel(1.0) * Matern(1.0, nu=1.5)
                       + WhiteKernel(0.1),
}
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
for ax, (name, k) in zip(axes, kernels.items()):
    g = GaussianProcessRegressor(kernel=k, normalize_y=True,
                                 n_restarts_optimizer=5,
                                 random_state=42).fit(X, y_obs)
    m, s = g.predict(x_grid, return_std=True)
    ax.plot(x_grid, m, color=VIZ['series'][0], label='прогноз')
    ax.fill_between(x_grid.ravel(), m - 2 * s, m + 2 * s,
                    color=VIZ['series'][0], alpha=0.2)
    ax.scatter(x_obs, y_obs, color=VIZ['series'][3], zorder=5)
    ax.set_title(name)
    ax.set_xlabel('Аргумент x')
    ax.set_ylabel('Значение функции')
fig.tight_layout()
plt.show()

**Как читать этот график.**

Два графика — одни данные, два ядра. Синяя линия — прогноз; голубая полоса — неопределённость. Обратите внимание на разницу в пробеле: RBF даёт более гладкую поверхность прогноза, Matern — менее гладкую. Если истинная функция имеет «углы» или быстро меняется — Matern предпочтительнее. Если физически ожидается плавная зависимость — RBF уместен.

## 5. Интерпретация результата

- **Прогноз и полоса неопределённости**: апостериорное среднее — наилучшая оценка; полоса ±2 стандартных отклонения — диапазон, где истинная функция лежит с вероятностью ~95 %. Полоса узкая рядом с наблюдениями и широкая в пробеле и за пределами данных.
- **Образцы функций**: каждый образец — правдоподобная гипотеза о форме функции; расхождение образцов — наглядная картина неопределённости. Это интуитивнее, чем полоса: видно, что в пробеле функция могла бы подняться, опуститься или пройти почти горизонтально.
- **Подобранное ядро**: параметр `length_scale` — масштаб изменчивости; `noise_level` (из `WhiteKernel`) — оценённая дисперсия шума измерений.
- **Выбор ядра — содержательное решение**: RBF предполагает очень гладкую функцию; Matern допускает менее регулярное поведение. Ядро отражает физические предположения о природе зависимости.
- **Ограничение метода**: вычисления растут как куб числа наблюдений, поэтому для выборок свыше нескольких тысяч точек нужны разреженные приближения.

## 6. Попробуйте на своих данных

Замените синтетические данные своими наблюдениями одномерной зависимости.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имена столбцов.
3. Выполните ячейки раздела 4.

## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры:

- **Размер пробела**: сдвиньте границы пробела с `[1, 3]` на `[−2, 2]` (широкий пробел в центре). Как изменится форма неопределённости? Смогла ли модель правильно экстраполировать через большой пробел?
- **Уровень шума**: измените `scale=0.2` на `0.05` (чистые данные) или `1.0` (шумные). Как изменится ширина полосы?
- **Число наблюдений**: уменьшите с 22 до 5 или увеличьте до 50. При каком числе точек неопределённость становится приемлемо малой?
- **Ядро**: добавьте `Periodic` ядро (`ExpSineSquared`) — подойдёт, если ваша функция периодическая. Какие изменения вы видите в прогнозе?
- **Нарушение предположений**: попробуйте разрывную функцию `true_function(x) = np.where(x < 0, -1.0, 1.0)`. Что делает гауссовский процесс с разрывом?

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# x_obs = df['аргумент'].to_numpy()
# y_obs = df['значение'].to_numpy()
# X = x_obs.reshape(-1, 1)
#
# Подберите диапазон сетки x_grid под свои данные.
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Гауссовский процесс — это **байесовская регрессия**: вместо одной кривой он строит распределение над всеми правдоподобными кривыми.
- Главный результат — не только прогноз, но и **честная мера неопределённости**: полоса узкая там, где данных много, и широкая там, где данных нет.
- **Ядро** задаёт предположения о гладкости и структуре зависимости — его выбор отражает физические знания о задаче.
- **Образцы функций** наглядно показывают, какой набор объяснений данных метод считает правдоподобными.
- Метод работает лучше всего при малых и средних выборках (до нескольких тысяч точек); при больших нужны разреженные аппроксимации.
- Гауссовский процесс — основа байесовской оптимизации и суррогатного моделирования: именно его неопределённость указывает, где ставить следующее измерение.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. В области пробела [1, 3] полоса неопределённости расширяется, а образцы функций расходятся. Однако все образцы остаются относительно гладкими кривыми. Что это говорит о подобранном ядре и как изменилась бы картина, если бы истинная функция имела разрыв?
2. Вы подобрали гауссовский процесс на своих данных, и параметр `noise_level` из `WhiteKernel` оказался равным 2.3, тогда как вы ожидали дисперсию шума измерений около 0.04. Назовите два возможных объяснения этого расхождения и что следует предпринять.
3. Коллега предлагает использовать гауссовский процесс с ядром RBF для прогнозирования сигнала прибора, который имеет известную периодичность 24 часа. Почему ядро RBF неоптимально для этой задачи, и какое ядро (или их комбинацию) следует использовать вместо него?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Образцы остаются гладкими потому, что ядро RBF предполагает бесконечно гладкие функции — это априорное предположение, встроенное в выбор ядра. При разрывной истинной функции гауссовский процесс с RBF-ядром будет давать широкую неопределённость вблизи разрыва, но никогда не воспроизведёт сам разрыв: все образцы останутся гладкими, а прогноз будет систематически смещён. В таком случае нужно изменить ядро или применить другой метод.
2. Возможные объяснения: (а) в данных присутствует систематический тренд или нелинейность, не описываемая выбранным ядром, — модель «списывает» необъяснённую вариацию на шум; (б) масштаб целевой переменной не нормирован, и параметры ядра оптимизируются в неудобном пространстве. Действия: проверить остатки модели (разность прогноза и наблюдений), нормировать `y_obs`, добавить полиномиальное или линейное ядро для тренда, проверить разумность `length_scale`.
3. Ядро RBF предполагает убывание корреляции с расстоянием и не кодирует периодичность. Для периодического сигнала используют ядро `ExpSineSquared` (периодическое ядро), которое задаёт похожесть точек, отстоящих на период. Для реального сигнала с затухающей периодичностью применяют произведение `RBF * ExpSineSquared` — это моделирует периодические колебания с убывающей со временем амплитудой.
</details>
